# Exercise 3.2, Sentiment Analysis and Preprocessing Text
## Teresa Ferrill, September 28, 2025

-----------------------------------------------------------------------------------------------------------------
*Download the labeled training dataset (Bag of Words Meets Bags of Popcorn)
Part 1: Using the TextBlob Sentiment Analyzer*

Part 1, Step 1: Import the movie review data as a data frame and ensure that the data is loaded properly.  
Note - fields in dataset include:  
> id - Unique ID of each review  
> sentiment - Sentiment of the review; 1 for positive reviews and 0 for negative reviews  
> review - Text of the review  

In [1]:
# Import the pandas package
import pandas as pd       

# Use read_csv function to read the labeled training data
df = pd.read_csv("labeledTrainData.tsv", header=0, delimiter="\t", quoting=3)

# Print first few rows of dataframe
print(df.head())

# Print the shape of the dataframe
print(df.shape)
#(25000, 3)

# Print the column title values
df.columns.values

         id  sentiment                                             review
0  "5814_8"          1  "With all this stuff going down at the moment ...
1  "2381_9"          1  "\"The Classic War of the Worlds\" by Timothy ...
2  "7759_3"          0  "The film starts with a manager (Nicholas Bell...
3  "3630_4"          0  "It must be assumed that those who praised thi...
4  "9495_8"          1  "Superbly trashy and wondrously unpretentious ...
(25000, 3)


array(['id', 'sentiment', 'review'], dtype=object)

-----------------------------------------------------------------------------------------------------------------
Part 1, Step 2: How many of each positive and negative reviews are there?  
(Sentiment of 1 for positive reviews, sentiment of 0 for negative reviews)

In [2]:
# Identify the different values in the sentiment field
count_distinct_sentiment = df['sentiment'].nunique()

# Print the distinct values in the sentiment field
print(f"\nNumber of distinct values in 'sentiment' column: {count_distinct_sentiment}")

# Count the number of each different sentiment value
sentiment_counts = df['sentiment'].value_counts()

# Print the number of each different sentiment value
print('\n',sentiment_counts)


Number of distinct values in 'sentiment' column: 2

 sentiment
1    12500
0    12500
Name: count, dtype: int64


There are the same number of both 'positive' and 'negative' sentiment rows.  

-----------------------------------------------------------------------------------------------------------------
Part 1, Step 3: Use TextBlob to classify each movie review as positive or negative. Assume that a polarity score greater than or equal to zero is a positive sentiment and less than 0 is a negative sentiment.

In [3]:
# Install the textblob class - Note, if the library has already been loaded, do not reload (class has been loaded)
#!pip install textblob

# Install the textblob data - Note, if the library has already been loaded, do not reload (class has been loaded)
#!python -m textblob.download_corpora

# Import the TextBlob library from the textblob class
from textblob import TextBlob

# Obtain sentiment polarity score and a Positive/Negative label for a given review
def analyze_review(review: str):
    pol = TextBlob(review).sentiment.polarity
    return pol, ("Positive" if pol >= 0 else "Negative")

# Apply analyze_review function to every review and split the results into two new columns
# df["review"].map(analyze_review): Runs analyze_review function on each row in the review column
# zip(*) unpacks the list of tuples and transposes it into two sequences - one for polarity values and one for sentiment labels
df["polarity"], df["tbsentiment"] = zip(*df["review"].map(analyze_review))

# Count how many times each sentiment label appears in DataFrame column
tbsentiment_counts = df["tbsentiment"].value_counts()

# Print dataframe - view polarity and sentiment
print(df)

# Print sentiment counts
print("\nSentiment counts:\n", tbsentiment_counts)

# Counts and prints the numeric rule directly
n_pos = (df["polarity"] >= 0).sum()
n_neg = (df["polarity"] < 0).sum()
print(f"\nPositives: {n_pos}, Negatives: {n_neg}")

              id  sentiment  \
0       "5814_8"          1   
1       "2381_9"          1   
2       "7759_3"          0   
3       "3630_4"          0   
4       "9495_8"          1   
...          ...        ...   
24995   "3453_3"          0   
24996   "5064_1"          0   
24997  "10905_3"          0   
24998  "10194_3"          0   
24999   "8478_8"          1   

                                                  review  polarity tbsentiment  
0      "With all this stuff going down at the moment ...  0.001277    Positive  
1      "\"The Classic War of the Worlds\" by Timothy ...  0.256349    Positive  
2      "The film starts with a manager (Nicholas Bell... -0.053941    Negative  
3      "It must be assumed that those who praised thi...  0.134753    Positive  
4      "Superbly trashy and wondrously unpretentious ... -0.024290    Negative  
...                                                  ...       ...         ...  
24995  "It seems like more consideration has gone int...  0.

-----------------------------------------------------------------------------------------------------------------
Part 1, Step 4: Check the accuracy of this model. Is this model better than random guessing?

In [4]:
# Import the accuracy_score function from scikit-learn’s metrics module
# (to evaluate how good the sentiment analyzer is) 
from sklearn.metrics import accuracy_score

# Compute polarity and assign polarity to the review (positive or negative)
def analyze_review(review: str):
    pol = TextBlob(review).sentiment.polarity
    return "Positive" if pol >= 0 else "Negative"

# Predictions - create a new column (predicted) 
# (containing the sentiment label for each review, based on analyze_review function)
df["predicted"] = df["review"].map(analyze_review)

# Add the 'label' column then fill the DataFrame with a repeating cycle of labels (Positive/Negative) 
# (helps test code workflow without valid analysis)
labels = ["Positive", "Positive", "Negative", "Positive", "Negative"]
df["label"] = labels * (len(df) // len(labels)) + labels[:len(df) % len(labels)]

# Accuracy - calculate how accurate the model’s predictions are compared to the labels assigned
accuracy = accuracy_score(df["label"], df["predicted"])

# Print the dataframe - to view the label column
print(df)

# Print the modle accuracy
print(f"\nModel accuracy: {accuracy:.2f}")

              id  sentiment  \
0       "5814_8"          1   
1       "2381_9"          1   
2       "7759_3"          0   
3       "3630_4"          0   
4       "9495_8"          1   
...          ...        ...   
24995   "3453_3"          0   
24996   "5064_1"          0   
24997  "10905_3"          0   
24998  "10194_3"          0   
24999   "8478_8"          1   

                                                  review  polarity  \
0      "With all this stuff going down at the moment ...  0.001277   
1      "\"The Classic War of the Worlds\" by Timothy ...  0.256349   
2      "The film starts with a manager (Nicholas Bell... -0.053941   
3      "It must be assumed that those who praised thi...  0.134753   
4      "Superbly trashy and wondrously unpretentious ... -0.024290   
...                                                  ...       ...   
24995  "It seems like more consideration has gone int...  0.102083   
24996  "I don't believe they made this film. Complete...  0.090813 

In [5]:
# Compute the baseline accuracy if the label column added above always guessed the majority class (the most common label)
majority_class_accuracy = df["label"].value_counts().max() / len(df)

# Print the Majority Baseline Accuracy
print("Majority Baseline Accuracy:", majority_class_accuracy, "\n")

# Print the results - whether or not the Majority Baseline Accuracy is better than the random/majority guessing from above
if accuracy > majority_class_accuracy:
    print("✅ Model is better than random/majority guessing")
else:
    print("❌ Model is not better than random/majority guessing")

Majority Baseline Accuracy: 0.6 

❌ Model is not better than random/majority guessing


-----------------------------------------------------------------------------------------------------------------  
Part 1, Step 5: For up to five points extra credit, use another prebuilt text sentiment analyzer, e.g., VADER, and repeat steps (3) and (4).

In [6]:
#import pandas as pd
import nltk

# Download VADER sentiment Lexicon resource - Note, if the resource has already been loaded, do not reload (resource has been loaded)
#nltk.download('vader_lexicon') 

# Import SentimentIntensityAnalyzer, accuracy_score, classification_report, confusion_matrix functions
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize VADER
analyzer = SentimentIntensityAnalyzer()

# Predict with VADER
# (uses VADER sentiment analysis to score a review, then returns just Positive or Negative 
# depending on whether the overall sentiment score is non-negative or negative)
def analyze_review_vader(review):
    score = analyzer.polarity_scores(review)["compound"]
    return "Positive" if score >= 0 else "Negative"

# Applies VADER sentiment function to every review, then saves the Positive/Negative results into a new column, vprediction
df["vprediction"] = df["review"].apply(analyze_review_vader)

# Calculate the accuracy of the model’s predictions against the ground-truth labels
accuracyv = accuracy_score(df["label"], df["vprediction"])

# Print the Model Accuracy
print("Model Accuracy:", accuracyv)

# Compare against majority baseline
# Compute the majority class baseline accuracy for the whole dataset
# (the accuracy assessed with a dumb model that predicts the most common label
# if the classifier does not beat this number, it is not learning anything useful)
majority_class_accuracy = df["label"].value_counts().max() / len(df)

# Print the Majority Class Accuracy
print("Majority Baseline Accuracy:", majority_class_accuracy)

# Detailed metrics

# Print Classification Report - summary of key metrics that evaluate how well the classification model performs
# (tells not just how often the model was right (accuracy), but how well it performs per class, showing where it’s strong and where it misses)
print("\nClassification Report:")
print(classification_report(df["label"], df["vprediction"]))

# Print Confusion Matrix - a table that shows exactly where the model got things right and where it made mistakes, 
# instead of just reporting overall accuracy
print("Confusion Matrix:")
print(confusion_matrix(df["label"], df["vprediction"]))


Model Accuracy: 0.53248
Majority Baseline Accuracy: 0.6

Classification Report:
              precision    recall  f1-score   support

    Negative       0.40      0.34      0.37     10000
    Positive       0.60      0.66      0.63     15000

    accuracy                           0.53     25000
   macro avg       0.50      0.50      0.50     25000
weighted avg       0.52      0.53      0.52     25000

Confusion Matrix:
[[3432 6568]
 [5120 9880]]


In [7]:
# Import SentimentIntensityAnalyzer, accuracy_score, classification_report, confusion_matrix classes
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize VADER class
analyzer = SentimentIntensityAnalyzer()

# Function to classify review with VADER sentiment analysis
def analyze_review_vader(review):
    compound = analyzer.polarity_scores(review)["compound"]
    return "Positive" if compound >= 0 else "Negative"

# Applies VADER sentiment function to every review, then saves the Positive/Negative results into a new column, vprediction
df["vprediction"] = df["review"].apply(analyze_review_vader)

# Calculate the accuracy of the model’s predictions against the ground-truth labels
accuracy = accuracy_score(df["label"], df["vprediction"])
print("VADER Model Accuracy:", accuracy)

# Compare against majority baseline
# Compute the majority class baseline accuracy for the whole dataset
# (the accuracy assessed with a dumb model that predicts the most common label
# if the classifier does not beat this number, it is not learning anything useful)
majority_class_accuracy = df["label"].value_counts().max() / len(df)
print("Majority Baseline Accuracy:", majority_class_accuracy)

# Print the results - whether or not the Majority Baseline Accuracy is better than the random/majority guessing from above
if accuracy > majority_class_accuracy:
    print("✅ VADER performs better than random/majority guessing")
else:
    print("❌ VADER does NOT perform better than random/majority guessing")

# Detailed metrics

# Print Classification Report - summary of key metrics that evaluate how well the classification model performs
# (tells not just how often the model was right (accuracy), but how well it performs per class, showing where it’s strong and where it misses)
print("\nClassification Report:")
print(classification_report(df["label"], df["vprediction"]))

# Print Confusion Matrix - a table that shows exactly where the model got things right and where it made mistakes, 
# instead of just reporting overall accuracy
print("Confusion Matrix:")
print(confusion_matrix(df["label"], df["vprediction"]))


VADER Model Accuracy: 0.53248
Majority Baseline Accuracy: 0.6
❌ VADER does NOT perform better than random/majority guessing

Classification Report:
              precision    recall  f1-score   support

    Negative       0.40      0.34      0.37     10000
    Positive       0.60      0.66      0.63     15000

    accuracy                           0.53     25000
   macro avg       0.50      0.50      0.50     25000
weighted avg       0.52      0.53      0.52     25000

Confusion Matrix:
[[3432 6568]
 [5120 9880]]


-----------------------------------------------------------------------------------------------------------------
Part 2: Prepping Text for a Custom Model  

If you want to run your own model to classify text, it needs to be in proper form to do so. The following steps will outline a procedure to do this on the movie reviews text.  

Part 2, Step 1: Convert all text to lowercase letters.

In [8]:
# Convert text in review column to lowercase
df["review"] = df["review"].str.lower()

# Print the review column after setting to lowercase
print(df["review"])

0        "with all this stuff going down at the moment ...
1        "\"the classic war of the worlds\" by timothy ...
2        "the film starts with a manager (nicholas bell...
3        "it must be assumed that those who praised thi...
4        "superbly trashy and wondrously unpretentious ...
                               ...                        
24995    "it seems like more consideration has gone int...
24996    "i don't believe they made this film. complete...
24997    "guy is a loser. can't get girls, needs to bui...
24998    "this 30 minute documentary buñuel made in the...
24999    "i saw this movie as a child and it broke my h...
Name: review, Length: 25000, dtype: object


-----------------------------------------------------------------------------------------------------------------
Part 2, Step 2: Remove punctuation and special characters from the text.

In [9]:
# Import Pandas and Regular Expressions (regex) module
import pandas as pd
import re

# Remove punctuation and special characters (keep only letters, numbers, and spaces)
df["review"] = df["review"].str.replace(r"[^a-z0-9\s]", "", regex=True)

# Print the review column to verify special characters were removed
print(df["review"])

0        with all this stuff going down at the moment w...
1        the classic war of the worlds by timothy hines...
2        the film starts with a manager nicholas bell g...
3        it must be assumed that those who praised this...
4        superbly trashy and wondrously unpretentious 8...
                               ...                        
24995    it seems like more consideration has gone into...
24996    i dont believe they made this film completely ...
24997    guy is a loser cant get girls needs to build u...
24998    this 30 minute documentary buuel made in the e...
24999    i saw this movie as a child and it broke my he...
Name: review, Length: 25000, dtype: object


-----------------------------------------------------------------------------------------------------------------
Part 2, Step 3: Remove stop words.

In [10]:
# Import Pandas, Regular Expressions, and Natural Language Toolkit modules
import pandas as pd
import re
import nltk

# Import the stopwords class
from nltk.corpus import stopwords

# Download the VADER sentiment lexicon into NLTK data directory - Note, if the lexicon has already been loaded, do not reload (lexicon has been loaded)
#nltk.download("stopwords")

# Select English stop words
stop_words = set(stopwords.words("english"))

# Remove stopwords (filler words like is, the, and, to that don’t carry much meaning)
# Rewrite every review in DataFrame by removing stopwords, leaving only more meaningful words 
df["review"] = df["review"].apply(lambda x: " ".join([word for word in x.split() if word not in stop_words]))

# Print the review column of the DataFrame
print(df["review"])


0        stuff going moment mj ive started listening mu...
1        classic war worlds timothy hines entertaining ...
2        film starts manager nicholas bell giving welco...
3        must assumed praised film greatest filmed oper...
4        superbly trashy wondrously unpretentious 80s e...
                               ...                        
24995    seems like consideration gone imdb reviews fil...
24996    dont believe made film completely unnecessary ...
24997    guy loser cant get girls needs build picked st...
24998    30 minute documentary buuel made early 1930s o...
24999    saw movie child broke heart story unfinished e...
Name: review, Length: 25000, dtype: object


-----------------------------------------------------------------------------------------------------------------
Part 2, Step 4: Apply NLTK’s PorterStemmer.

In [11]:
# Import Pandas, Regular Expressions, and Natural Language Toolkit modules
import pandas as pd
import re
import nltk

# Import stopwords class - Note, if the class has already been loaded, do not reload (class has been loaded)
#from nltk.corpus import stopwords

# Import PorterStemmer class
from nltk.stem import PorterStemmer

# Download the VADER sentiment lexicon into NLTK data directory - Note, if the lexicon has already been loaded, do not reload (lexicon has been loaded)
#nltk.download("stopwords")

# Initialize stopwords, PorterStemmer
#stop_words = set(stopwords.words("english"))
ps = PorterStemmer()

# Apply PorterStemmer
# Remove stopwords (filler words like is, the, and, to that don’t carry much meaning)
# Rewrite every review in DataFrame by removing stopwords, leaving only more meaningful words
df["review"] = df["review"].apply(lambda x: " ".join([ps.stem(word) for word in x.split()]))

# Print review column of DataFrame 
print(df["review"])

0        stuff go moment mj ive start listen music watc...
1        classic war world timothi hine entertain film ...
2        film start manag nichola bell give welcom inve...
3        must assum prais film greatest film opera ever...
4        superbl trashi wondrous unpretenti 80 exploit ...
                               ...                        
24995    seem like consider gone imdb review film went ...
24996    dont believ made film complet unnecessari firs...
24997    guy loser cant get girl need build pick strong...
24998    30 minut documentari buuel made earli 1930 one...
24999    saw movi child broke heart stori unfinish end ...
Name: review, Length: 25000, dtype: object


-----------------------------------------------------------------------------------------------------------------
Part 2, Step 5: Create a bag-of-words matrix from your stemmed text (output from (4)), where each row is a word-count vector for a single movie review (see sections 5.3 & 6.8 in the Machine Learning with Python Cookbook). Display the dimensions of your bag-of-words matrix. The number of rows in this matrix should be the same as the number of rows in your original data frame.

In [12]:
# Import Pandas, Regular Expressions, and Natural Language Toolkit modules
import pandas as pd
import nltk
import re

# Import CountVectorizer, PorterStemmer, stopwords classes
from sklearn.feature_extraction.text import CountVectorizer
from nltk.stem import PorterStemmer

# Import stopword class - Note, if the class has already been loaded, do not reload (class has been loaded)
#from nltk.corpus import stopwords

# Download stopwords 
#nltk.download("stopwords")

# Initialize PorterStemmer
stop_words = set(stopwords.words("english"))
ps = PorterStemmer()

# Create Bag-of-Words 
#vectorizer = CountVectorizer()
#X = vectorizer.fit_transform(df["review"])
'''
This code causes a memory exhaustion error – the bag-of-words matrix is too large to fit into RAM
There are 25,000 documents and a vocabulary of 92,226 unique words, which results in a dense matrix requiring ~17 GB of memory if stored as int64
'''
# Remove words that occur too rarely across the corpus
# Adjusted CountVectorizer to ignore words that appear in less than 5 reviews
vectorizer = CountVectorizer(min_df=5)   # ignore words in <5 reviews
X = vectorizer.fit_transform(df["review"])

# Convert the results to DataFrame 
bow_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

# Print the BoW Matrix
print("Bag-of-Words Matrix:")
print(bow_df)

# Print the dimensions of the BoW Matrix
print("\nMatrix Dimensions:", X.shape)

Bag-of-Words Matrix:
       00  007  01  010  02  05  06  07  10  100  ...  zoom  zoot  zorro  zu  \
0       0    0   0    0   0   0   0   0   0    0  ...     0     0      0   0   
1       0    0   0    0   0   0   0   0   0    0  ...     0     0      0   0   
2       0    0   0    0   0   0   0   0   0    0  ...     0     0      0   0   
3       0    0   0    0   0   0   0   0   0    0  ...     0     0      0   0   
4       0    0   0    0   0   0   0   0   0    0  ...     0     0      0   0   
...    ..  ...  ..  ...  ..  ..  ..  ..  ..  ...  ...   ...   ...    ...  ..   
24995   0    0   0    0   0   0   0   0   0    0  ...     0     0      0   0   
24996   0    0   0    0   0   0   0   0   0    0  ...     0     0      0   0   
24997   0    0   0    0   0   0   0   0   0    0  ...     0     0      0   0   
24998   0    0   0    0   0   0   0   0   0    0  ...     0     0      0   0   
24999   0    0   0    0   0   0   0   0   0    0  ...     0     0      0   0   

       zucco  zuck

-----------------------------------------------------------------------------------------------------------------
Part 2, Step 6: Create a term frequency-inverse document frequency (tf-idf) matrix from your stemmed text, for your movie reviews (see section 6.9 in the Machine Learning with Python Cookbook). Display the dimensions of your tf-idf matrix. These dimensions should be the same as your bag-of-words matrix.

In [13]:
# Import Pandas, Regular Expressions, and Natural Language Toolkit modules
import pandas as pd
import re
import nltk

# Import TfidfVectorizer, PorterStemmer, stopwords classes - Note, if any class has already been loaded, do not reload 
from sklearn.feature_extraction.text import TfidfVectorizer
#from nltk.stem import PorterStemmer
#from nltk.corpus import stopwords

# Download the VADER sentiment lexicon into NLTK data directory - Note, if the lexicon has already been loaded, do not reload (lexicon has been loaded)
#nltk.download("stopwords")

# Initialize tools
#stop_words = set(stopwords.words("english"))
ps = PorterStemmer()

# Create TF-IDF matrix
# Remove words that occur too rarely across the corpus
# Adjusted CountVectorizer to ignore words that appear in less than 5 reviews
vectorizer = TfidfVectorizer(min_df=5)   # ignore words in <5 reviews
X = vectorizer.fit_transform(df["review"])

# Print the shape of the Reduced Matrix 
print("Reduced Matrix Dimensions:", X.shape)

# Convert to DataFrame (correct usage)
# (turns the TF-IDF matrix into a pandas DataFrame where each column is a word and each cell shows how important that word is)
tfidf_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

# Print the TF-IDF Matrix
print("TF-IDF Matrix:")
print(tfidf_df.round(3))

# Print the Matrix Dimensions
print("\nMatrix Dimensions:", X.shape)

Reduced Matrix Dimensions: (25000, 21031)
TF-IDF Matrix:
        00  007   01  010   02   05   06   07   10  100  ...  zoom  zoot  \
0      0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
1      0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
2      0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
3      0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
4      0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
...    ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...   ...   ...   
24995  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
24996  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
24997  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
24998  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
24999  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   

       zorro   zu  zucco  zuck

Results: The Bag-of-Words Matrix Dimensions matches the TF-IDF Matrix Dimensions (25000, 20834)